In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier   
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings("ignore")

In [17]:
def rfeFeature(in_x,out_y,n):
        rfelist=[]
        
        log_model = LogisticRegression(solver='lbfgs')
        RF = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
       # NB = GaussianNB()
        DT= DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
        svc_model = SVC(kernel = 'linear', random_state = 0)
        #knn = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        rfemodellist=[log_model,svc_model,RF,DT] 
        for i in   rfemodellist:
            print(i)
            log_rfe = RFE(estimator=i, n_features_to_select=n)
            #log_rfe = RFE(i, n)
            log_fit = log_rfe.fit(in_x, out_y)
            log_rfe_feature=log_fit.transform(in_x)
            rfelist.append(log_rfe_feature)
            print("Selected features:", in_x.columns[log_rfe.get_support()])
        return rfelist

In [18]:
def split_scalar(in_x,out_y):
        x_train, x_test, y_train, y_test = train_test_split(in_x, out_y, test_size = 0.25, random_state = 0)
        #X_train, X_test, y_train, y_test = train_test_split(indep_X,dep_Y, test_size = 0.25, random_state = 0)
        
        #Feature Scaling
        #from sklearn.preprocessing import StandardScaler
        sc = StandardScaler()
        X_train = sc.fit_transform(x_train)
        X_test = sc.transform(x_test)
        
        return x_train, x_test, y_train, y_test

In [19]:
def cm_prediction(classifier,x_test):
     y_pred = classifier.predict(x_test)
        
        # Making the Confusion Matrix
     from sklearn.metrics import confusion_matrix
     cm = confusion_matrix(y_test, y_pred)
        
     from sklearn.metrics import accuracy_score 
     from sklearn.metrics import classification_report 
        
     Accuracy=accuracy_score(y_test, y_pred )
        
     report=classification_report(y_test, y_pred)
     return  classifier,Accuracy,report,x_test,y_test,cm

In [20]:
def logistic(x_train,y_train,x_test):       

        from sklearn.linear_model import LogisticRegression
        classifier = LogisticRegression(random_state = 0)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm      

In [21]:
def svm_linear(x_train,y_train,x_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'linear', random_state = 0)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm

In [22]:
def svm_NL(x_train,y_train,x_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'rbf', random_state = 0)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm

In [23]:
def Navie(x_train,y_train,x_test):       

        from sklearn.naive_bayes import GaussianNB
        classifier = GaussianNB()
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm         

In [24]:
def knn(x_train,y_train,x_test):
           

        from sklearn.neighbors import KNeighborsClassifier
        classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm

In [25]:
def Decision(x_train,y_train,x_test):
        

        from sklearn.tree import DecisionTreeClassifier
        classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm      

In [26]:
def random(x_train,y_train,x_test):
        

        from sklearn.ensemble import RandomForestClassifier
        classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
        classifier.fit(x_train, y_train)
        classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
        return  classifier,Accuracy,report,x_test,y_test,cm

In [27]:
def rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf): 
    
    rfedataframe=pd.DataFrame(index=['Logistic','SVC','Random','DecisionTree'],columns=['Logistic','SVMl','SVMNl',
                                                                                        'KNN','Navie','Decision','Random'])

    for number,idex in enumerate(rfedataframe.index):
        
        rfedataframe['Logistic'][idex]=acclog[number]       
        rfedataframe['SVMl'][idex]=accsvml[number]
        rfedataframe['SVMNl'][idex]=accsvmnl[number]
        rfedataframe['KNN'][idex]=accknn[number]
        rfedataframe['Navie'][idex]=accnav[number]
        rfedataframe['Decision'][idex]=accdes[number]
        rfedataframe['Random'][idex]=accrf[number]
    return rfedataframe

In [30]:
dataset1=pd.read_csv("Stroke_Prepdata.csv",index_col=None)

df=pd.get_dummies(dataset1,drop_first=True)

in_x=df[["age","gender_Male","hypertension","heart_disease","avg_glucose_level","bmi","ever_married_Yes","work_type_Never_worked","work_type_Private","work_type_Self-employed","work_type_children","Residence_type_Urban","smoking_status_formerly smoked","smoking_status_never smoked","smoking_status_smokes",]]
out_y=df['stroke']

rfelist=rfeFeature(in_x,out_y,4)  
rfelist

LogisticRegression()
Selected features: Index(['hypertension', 'heart_disease', 'work_type_children',
       'smoking_status_formerly smoked'],
      dtype='object')
SVC(kernel='linear', random_state=0)
Selected features: Index(['work_type_Self-employed', 'smoking_status_formerly smoked',
       'smoking_status_never smoked', 'smoking_status_smokes'],
      dtype='object')
RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)
Selected features: Index(['age', 'avg_glucose_level', 'bmi', 'work_type_Private'], dtype='object')
DecisionTreeClassifier(max_features='sqrt', random_state=0)
Selected features: Index(['age', 'heart_disease', 'avg_glucose_level', 'bmi'], dtype='object')


[array([[0., 1., 0., 1.],
        [0., 0., 0., 0.],
        [0., 1., 0., 0.],
        ...,
        [0., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 0., 0., 0.]]),
 array([[0., 1., 0., 0.],
        [1., 0., 1., 0.],
        [0., 0., 1., 0.],
        ...,
        [1., 0., 1., 0.],
        [0., 1., 0., 0.],
        [0., 0., 0., 0.]]),
 array([[ 67.        , 228.69      ,  36.6       ,   1.        ],
        [ 61.        , 202.21      ,  28.89323691,   0.        ],
        [ 80.        , 105.92      ,  32.5       ,   1.        ],
        ...,
        [ 35.        ,  82.99      ,  30.6       ,   0.        ],
        [ 51.        , 166.29      ,  25.6       ,   1.        ],
        [ 44.        ,  85.28      ,  26.2       ,   0.        ]]),
 array([[ 67.        ,   1.        , 228.69      ,  36.6       ],
        [ 61.        ,   0.        , 202.21      ,  28.89323691],
        [ 80.        ,   1.        , 105.92      ,  32.5       ],
        ...,
        [ 35.        ,   0.        ,

In [31]:
acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

for i in rfelist:   
    x_train, x_test, y_train, y_test=split_scalar(i,out_y)   
    
        
    classifier,Accuracy,report,x_test,y_test,cm=logistic(x_train,y_train,x_test)
    acclog.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=svm_linear(x_train,y_train,x_test)  
    accsvml.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=svm_NL(x_train,y_train,x_test)  
    accsvmnl.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=knn(x_train,y_train,x_test)  
    accknn.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=Navie(x_train,y_train,x_test)  
    accnav.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=Decision(x_train,y_train,x_test)  
    accdes.append(Accuracy)
    
    classifier,Accuracy,report,x_test,y_test,cm=random(x_train,y_train,x_test)  
    accrf.append(Accuracy)
    
result=rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)

result

,Logistic,SVMl,SVMNl,KNN,Navie,Decision,Random
Logistic,0.949139,0.949139,0.949139,0.949139,0.181534,0.949139,0.949139
SVC,0.949139,0.949139,0.949139,0.949139,0.949139,0.949139,0.949139
Random,0.949139,0.949139,0.949139,0.944444,0.926448,0.919405,0.947574
DecisionTree,0.949139,0.949139,0.949139,0.944444,0.904538,0.920188,0.949139
